In [1]:
!pip install selenium


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: C:\Users\5g\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1080")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

all_collections = []

driver.get("https://egymonuments.gov.eg/en/collections")

# Waiting for the collection links to load
wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#cd-museums a")))
i = 1
while True:
    try:
        # press the next site link
        try:
            site_link = wait.until(EC.element_to_be_clickable(
            (By.XPATH, f'//*[@id="cd-museums"]/app-egyptian-treasure/div/div[3]/a[{i}]')
            ))
            site_link.click()
            i += 1
        except:
            driver.find_element(By.XPATH, '//*[@id="cd-museums"]/app-egyptian-treasure/div/div[3]/div/button').click()
            continue

        # collect the data
        place_name = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div.mainPageTitle'))).text
        place_location = driver.find_element(By.CSS_SELECTOR, 
            'div.relative.descriptionImageCont > div.DescriptionSection > div > div.itemInfo'
        ).text
        place_description = driver.find_element(By.CSS_SELECTOR, 
            'div.relative.descriptionImageCont > div.DescriptionSection > div > div.txtSection'
        ).text

        try:
            place_photo = driver.find_element(By.CSS_SELECTOR,
                'body > div.innerMainBgCont > div > div:nth-child(1) > div > div.relative.descriptionImageCont > div.imageCont img'
            ).get_attribute('src')
        except:
            place_photo = ""

        all_collections.append({
            "collection": place_name,
            "location": place_location,
            "Description": place_description,
            "photo_url": place_photo,
        })

        # previous page
        driver.back()
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#cd-museums a")))


    except Exception as e:
        print("Finished — no more items.")
        break

print(f"Total collections: {len(all_collections)}")

Finished — no more items.
Total collections: 159


In [6]:
df = pd.DataFrame(all_collections)
df

,collection,location,Description,photo_url
0,statue of ptolemaic queen in Isis attire,Graeco-Roman Museum,statue of a Ptolemaic queen in Isis attire \nt...,https://egymonuments.gov.eg/media/8923/picture...
1,statue of the goddess Isis,Marsa Matrouh Museum,Seated statue of the goddess Isis nursing Harp...,https://egymonuments.gov.eg/media/8911/picture...
2,A handle,Marsa Matrouh Museum,Fragment of a handle depicting two faces of Ha...,https://egymonuments.gov.eg/media/8910/picture...
3,statue of cat,Marsa Matrouh Museum,"statue of cat, thought to represent the goddes...",https://egymonuments.gov.eg/media/8909/picture...
4,Pottery vessel,Marsa Matrouh Museum,Reconstructed and restored pottery vessel with...,https://egymonuments.gov.eg/media/8907/picture...
...,...,...,...,...
154,Historical Cairo,Cairo,City of the Thousand Minarets\nGeneral Gawhar ...,https://egymonuments.gov.eg/media/6748/ai_imag...
155,Thebes,Luxor,Throne of God Amun\nThe astounding number of w...,https://egymonuments.gov.eg/media/6754/1612173...
156,Alexandria,Alexandria,The Pearl of the Mediterranean\nAlexandria wa...,https://egymonuments.gov.eg/media/6750/1612174...
157,Amarna,Al-Minya,The First Capital of Monotheism\nKing Amenhot...,https://egymonuments.gov.eg/media/6751/1612174...


In [7]:
df.to_csv('collections_en.csv', index=False)